# ENTSO-E Data Cleaning

The input is the merged range `2022-06-09_2026-10-21` from `02b_merging_ranges`, for PL, CZ, DE_LU, LT, SE_4 and SK. Duplicated price timestamps and the columns that are not reported in every download range (for example `Wind Offshore` in PL) were already removed in that step, so only one zone still needs cleaning.

### Lithuania (LT)

**Generation Forecast:** Exclude `generation_forecast` entirely due to insufficient data quality.

In [1]:
import pandas as pd
import logging
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent.parent))

from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
)
from src.utils.data_quality import print_missing_values, print_index_summary

from src.utils.cleaning import (
    clean_lt,
)
from utils.file_io import inspect_entsoe_files, save_entsoe_data, load_entsoe_data

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
logging.basicConfig(level=logging.INFO)

In [3]:
data_entsoe = load_entsoe_data(
    DATA_DIR / "raw" / "entsoe",
    date_range,
)

In [4]:
# Not used anymore: PL and DE_LU need no cleaning after the ranges are merged (02b), FR is not part of the dataset.
# data_entsoe["PL"] = clean_pl(data_entsoe["PL"])
# data_entsoe["DE_LU"] = clean_de_lu(data_entsoe["DE_LU"])
# data_entsoe["FR"] = clean_fr(data_entsoe["FR"])
data_entsoe["LT"] = clean_lt(data_entsoe["LT"])

# codes = ["PL", "DE_LU", "FR", "LT"]
codes = list(data_entsoe)

## 1. Index Integrity – Confirmation

None of the datasets contains duplicated index values. The only repeated timestamps come from the Summer/Winter time switch. Gaps in the time index are not part of this step and are handled later, during preprocessing.

In [5]:
for code in codes:
    print(f"===== {code} {"=" * 20}")
    for n, d in data_entsoe[code].items():
        print_index_summary(d, n)

===== CZ ====================

--- [load_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 10
Timezone:   Europe/Prague

Time steps:
  0 days 00:15:00: 80919
  0 days 01:00:00: 18050

--- [day_ahead_prices] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 7
Timezone:   Europe/Prague

Time steps:
  0 days 00:15:00: 36958
  0 days 01:00:00: 29036
  0 days 02:00:00: 2
  0 days 00:30:00: 1

--- [generation_wind_solar_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 10
Timezone:   Europe/Prague

Time steps:
  0 days 00:15:00: 80919
  0 days 01:00:00: 18050

--- [generation_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 10
Timezone:   Europe/Prague

Time steps:
  0 days 00:15:00: 80919
  0 days 01:00:00: 18050
===== DE_LU ====================

--- [load_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 16
Timezone:

## 2. Missing Values – Confirmation

No missing values remain in any dataset except `Solar` and `Wind Onshore` in the DE_LU `generation_wind_solar_forecast` (12 values each). They are DST-related short gaps to be filled during preprocessing.

In [6]:
for code in codes:
    print(f"===== {code} {"=" * 20}")
    for n, d in data_entsoe[code].items():
        print_missing_values(d, n)

===== CZ ====================

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
No missing values.

--- [generation_forecast] Missing values ---
No missing values.
===== DE_LU ====================

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
Solar           12
Wind Onshore    12

--- [generation_forecast] Missing values ---
No missing values.
===== LT ====================

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
No missing values.
===== PL ====================

--- [load] Missing values ---
No missing values.

--- [generation] Missing values ---
No missing values.

--- [load_forecast] Missing va

## 3. Conclusions

The cleaning decision from the data quality audit has been applied to LT, and the checks above confirm the result. The datasets of all bidding zones (PL, CZ, DE_LU, LT, SE_4 and SK), including those that required no cleaning, are saved to `data/clean/entsoe/` and verified by reading the files back.

## 4. Saving Clean Data

In [7]:
save_entsoe_data(
    data_entsoe,
    DATA_DIR / "clean" / "entsoe",
    date_range,
)

INFO:utils.file_io:Saved clean/entsoe/CZ/2022-06-09_2026-10-21/load_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/CZ/2022-06-09_2026-10-21/day_ahead_prices.parquet
INFO:utils.file_io:Saved clean/entsoe/CZ/2022-06-09_2026-10-21/generation_wind_solar_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/CZ/2022-06-09_2026-10-21/generation_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/DE_LU/2022-06-09_2026-10-21/load_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/DE_LU/2022-06-09_2026-10-21/day_ahead_prices.parquet
INFO:utils.file_io:Saved clean/entsoe/DE_LU/2022-06-09_2026-10-21/generation_wind_solar_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/DE_LU/2022-06-09_2026-10-21/generation_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/LT/2022-06-09_2026-10-21/load_forecast.parquet
INFO:utils.file_io:Saved clean/entsoe/LT/2022-06-09_2026-10-21/day_ahead_prices.parquet
INFO:utils.file_io:Saved clean/entsoe/LT/2022-06-09_2026-10-21/generation_wind_sola

In [8]:
inspect_entsoe_files(
    DATA_DIR / "clean" / "entsoe",
    date_range,
)


--- CZ (4 file/s) ------------------------------
  [FILE] data/clean/entsoe/CZ/2022-06-09_2026-10-21/load_forecast.parquet | shape: (98970, 1) | idx: 2022-06-09 00:00:00+02:00-2026-10-20 23:45:00+02:00
  [FILE] data/clean/entsoe/CZ/2022-06-09_2026-10-21/day_ahead_prices.parquet | shape: (65998, 1) | idx: 2022-06-09 00:00:00+02:00-2026-10-21 00:00:00+02:00
  [FILE] data/clean/entsoe/CZ/2022-06-09_2026-10-21/generation_wind_solar_forecast.parquet | shape: (98970, 1) | idx: 2022-06-09 00:00:00+02:00-2026-10-20 23:45:00+02:00
  [FILE] data/clean/entsoe/CZ/2022-06-09_2026-10-21/generation_forecast.parquet | shape: (98970, 2) | idx: 2022-06-09 00:00:00+02:00-2026-10-20 23:45:00+02:00

--- DE_LU (4 file/s) ------------------------------
  [FILE] data/clean/entsoe/DE_LU/2022-06-09_2026-10-21/load_forecast.parquet | shape: (153112, 1) | idx: 2022-06-09 00:00:00+02:00-2026-10-20 23:45:00+02:00
  [FILE] data/clean/entsoe/DE_LU/2022-06-09_2026-10-21/day_ahead_prices.parquet | shape: (65998, 1) | 